In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [15]:

# Load dataset
# df = pd.read_csv("students_anxiety_depression.csv")
df = pd.read_excel("dataset.xlsx")
# df["label"] = df["label"].astype(int)
# Use a nullable integer type (note the capital 'I')
df["label"] = df["label"].astype('Int64')
df.tail()

,text,label
6977,I can't forget you #SpiritHadrian,0
6978,€ ®šæœŸâ˜†ã€'..DJ DAIKI! DJ DAIKI! DJ DAIKI!.D...,0
6979,Dai5y! <3,0
6980,tired of clowns but still hopefully tonight if...,0
6981,MW SUBI WN LA VACA,0


In [16]:
# Encode target
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])

# Split
X = df.drop('label', axis=1)
y = df['label']

# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Train model
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

ValueError: could not convert string to float: 'oh my gosh'

In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load dataset
# df = pd.read_csv("mental_health_text.csv")
df = pd.read_excel("dataset.xlsx")
# df["label"] = df["label"].astype(int)
# Use a nullable integer type (note the capital 'I')
df["label"] = df["label"].astype('Int64')
df.tail()

# Split data
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], test_size=0.2, random_state=42)

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Evaluate
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))


ValueError: np.nan is an invalid document, expected byte or unicode string.

In [30]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Load dataset
# df = pd.read_csv("mental_health_text.csv")
df = pd.read_excel("dataset.xlsx")
# df["label"] = df["label"].astype(int)
# Use a nullable integer type (note the capital 'I')
df["label"] = df["label"].astype('Int64')
df.tail()



,text,label
6977,I can't forget you #SpiritHadrian,0
6978,€ ®šæœŸâ˜†ã€'..DJ DAIKI! DJ DAIKI! DJ DAIKI!.D...,0
6979,Dai5y! <3,0
6980,tired of clowns but still hopefully tonight if...,0
6981,MW SUBI WN LA VACA,0


In [31]:
df.shape

(6982, 2)

In [ ]:

# ----- CLEANING -----
df['text'] = df['text'].fillna("").astype(str)               # replace NaN with empty string
df['text'] = df['text'].str.strip()                          # remove surrounding whitespace
df = df[~df['text'].isna()]                                  # just in case
# optionally drop truly empty texts:
df = df[df['text'] != ""]

# ----- SPLIT -----
X = df['text']
df["label"].isnull().sum()
df.isnull().sum()

df = df.dropna(axis=0)
# y = df['label'].astype(int)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ----- VECTORIZE -----
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# ----- HANDLE CLASS WEIGHT (simple) -----
classes = sorted(y.unique())
class_weights = dict(enumerate(compute_class_weight('balanced', classes=classes, y=y)))
print("class_weights:", class_weights)

# ----- TRAIN -----
model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
# or: model = LogisticRegression(max_iter=1000, class_weight=class_weights)
model.fit(X_train_tfidf, y_train)

# ----- EVALUATE -----
y_pred = model.predict(X_test_tfidf)
print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


In [24]:
print("df.shape:", df.shape)
print("columns:", df.columns.tolist())

# If X and y already exist:
print("len(X):", len(X))
print("len(y):", len(y))


df.shape: (6972, 1)
columns: ['text']
len(X): 6972
len(y): 6982


In [28]:
# 1) cleaning: drop columns that contain ANY NaN
df = df.dropna(axis=1, how='any')

# 2) sanity check: make sure the needed columns exist
required = ['text', 'label']
for col in required:
    if col not in df.columns:
        raise ValueError(f"Required column '{col}' not found after dropping columns. Available: {df.columns.tolist()}")

# 3) optional: remove empty text rows (if any)
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ""]   # remove truly empty strings if you want

# 4) now create X and y (AFTER cleaning)
X = df['text']
y = df['label'].astype(int)

# 5) debug shapes
print("After cleaning -> df.shape:", df.shape)
print("len(X):", len(X), "len(y):", len(y))

# 6) split (with stratify if classes present)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


ValueError: Required column 'label' not found after dropping columns. Available: ['text']

In [26]:
df.head()

,text
0,oh my gosh
1,"trouble sleeping, confused mind, restless hear..."
2,"All wrong, back off dear, forward doubt. Stay ..."
3,I've shifted my focus to something else but I'...
4,"I'm restless and restless, it's been a month n..."


In [9]:
# ====== IMPORTS ======
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# ====== LOAD DATA ======
# df = pd.read_csv("students_anxiety_depression.csv")   # change to your filename if needed
df = pd.read_excel("dataset.xlsx")
# ====== CLEANING ======
# Drop any row that has at least one NULL value
df = df.dropna(axis=0, how='any')

# Convert text to string and remove empty rows
df['text'] = df['text'].astype(str).str.strip()
df = df[df['text'] != ""]

# Check unique labels
print("Label distribution:\n", df['label'].value_counts())

# ====== FEATURES & TARGET ======
X = df['text']
y = df['label'].astype(int)   # assuming label is 0=normal, 1=anxiety/depression

# ====== TRAIN/TEST SPLIT ======
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ====== TF-IDF VECTORIZATION ======
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ====== MODEL TRAINING ======
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# ====== EVALUATION ======
y_pred = model.predict(X_test_tfidf)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Label distribution:
 label
0.0    6240
1.0     730
Name: count, dtype: int64

Accuracy: 0.9598278335724534

Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1248
           1       0.99      0.62      0.76       146

    accuracy                           0.96      1394
   macro avg       0.97      0.81      0.87      1394
weighted avg       0.96      0.96      0.96      1394



In [8]:
# ====== CUSTOM INPUT CHECK ======
# a =0
while True:
    user_input = input("\nEnter text to check (or type 'exit' to stop): ").strip()
    if user_input.lower() == 'exit':
        break

    # Convert input text to TF-IDF vector
    user_vec = tfidf.transform([user_input])

    # Predict
    pred = model.predict(user_vec)[0]
    prob = model.predict_proba(user_vec)[0]

    # Show result
    label = "Anxiety/Depression" if pred == 1 else "Normal"
    print(f"Predicted Label: {label}")
    print(f"Confidence: {max(prob)*100:.2f}%")
    # a+=1


NameError: name 'tfidf' is not defined

In [10]:
# ====== INTERACTIVE MULTI-QUESTION PREDICTOR ======
# Place this after you have trained `tfidf` and `model` in memory.

def ask_and_predict(model, tfidf):
    """
    Ask a few questions, concatenate answers, and run prediction.
    model: trained sklearn classifier with predict and predict_proba
    tfidf: fitted TfidfVectorizer
    """
    questions = [
        "1) How have you been feeling lately? (short sentence)",
        "2) Any trouble sleeping or changes in appetite? (yes/no or short)",
        "3) Are you finding it hard to enjoy things you used to? (yes/no or short)"
    ]

    print("\nAnswer the following questions. Type 'exit' at any time to quit.\n")

    while True:
        answers = []
        for q in questions:
            ans = input(q + "\n> ").strip()
            if ans.lower() == 'exit':
                print("Exiting interactive predictor.")
                return
            # ensure non-empty (allow short answers)
            if ans == "":
                ans = "no answer"
            answers.append(ans)

        # Combine answers into single string for model input
        combined_text = " ".join(answers)
        # Vectorize with the same TF-IDF used during training
        X_user = tfidf.transform([combined_text])

        # Do prediction
        try:
            pred = model.predict(X_user)[0]
        except Exception as e:
            print("Prediction failed:", e)
            continue

        # Confidence: if model supports predict_proba
        conf_text = ""
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(X_user)[0]
            confidence = max(proba)
            conf_text = f"Confidence: {confidence*100:.2f}%"
        else:
            conf_text = "Confidence: (model does not support predict_proba)"

        label_text = "Anxiety/Depression" if int(pred) == 1 else "Normal"

        print("\n=== Prediction Result ===")
        print(f"Predicted label: {label_text}")
        print(conf_text)
        print("=========================\n")

        # Optionally ask to continue or exit
        cont = input("Check another (y/n)? ").strip().lower()
        if cont not in ("y", "yes"):
            print("Stopping interactive predictor.")
            break

# Example usage: (uncomment to run)
ask_and_predict(model, tfidf)



Answer the following questions. Type 'exit' at any time to quit.


=== Prediction Result ===
Predicted label: Normal
Confidence: 93.00%


=== Prediction Result ===
Predicted label: Normal
Confidence: 94.79%


=== Prediction Result ===
Predicted label: Normal
Confidence: 96.16%


=== Prediction Result ===
Predicted label: Normal
Confidence: 96.06%

Stopping interactive predictor.
